# Mini LLM Inference Engine — run on Colab

This notebook only *runs* the project. All the code lives in the Python package in the repo.

1. **Runtime → Change runtime type → T4 GPU** (the project is designed for a T4).
2. **Runtime → Run all.**

Every cell is safe to run again after a disconnect: the setup cell clones the repo only once and pulls on later runs.

In [ ]:
# Setup (idempotent): clone or update the repo, then install it.
import os

REPO_URL = "https://github.com/Ham12-3/mini-inference.git"
REPO_DIR = "/content/mini-inference"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
# This checkout is a read-only mirror of GitHub: benchmark runs write result
# files into it, so update with fetch + hard reset rather than pull, which
# would refuse to overwrite them.
!git fetch -q && git reset -q --hard origin/master
!git log --oneline -1

# Lower-bound requirements only, so Colab's preinstalled torch is reused.
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

## Stage 0: GPU check and model config

In [ ]:
!nvidia-smi
!python -m benchmarks.env_info --model Qwen/Qwen2.5-0.5B-Instruct

## Stage 1: naive baseline (no KV cache)

Every new token re-runs the whole sequence through the model. About 2.5 minutes.

In [ ]:
!python -m benchmarks.single_request --backend naive --lengths 64 128 256 --repeats 3
!python -m benchmarks.single_request --backend naive --lengths 512 1024 --repeats 1 --name single_naive_long

## Stage 2: my Qwen2 model with a KV cache

Parity tests against Hugging Face, then single-request speed for my engine and for HF `generate()`.

In [ ]:
!python -m pytest tests/test_model_parity.py -q -p no:warnings

In [ ]:
!python -m benchmarks.single_request --backend engine_kv --lengths 64 128 256 512 1024 --repeats 3
!python -m benchmarks.single_request --backend hf_kv --lengths 64 128 256 512 1024 --repeats 3

## Stage 3: slot KV cache and static batching

Batched outputs must equal single-sequence outputs. Then throughput at batch sizes 1 to 32, for my engine and HF `generate()`.

In [ ]:
!python -m pytest tests/test_kv_cache.py -q -p no:warnings

In [ ]:
!python -m benchmarks.static_batch --backend engine
!python -m benchmarks.static_batch --backend engine_serial_prefill
!python -m benchmarks.static_batch --backend hf

## Stage 4: scheduler and continuous batching

Scheduler and sampler unit tests, then engine parity: requests arriving at different times must generate exactly what they generate alone.

In [ ]:
!python -m pytest tests/test_scheduler.py tests/test_sampler.py tests/test_engine_parity.py -q -p no:warnings

100 requests of mixed lengths through the engine. The debug log prints one line per step; requests join and leave the running batch as they arrive and finish.

In [ ]:
!python scripts/demo_continuous_batching.py --debug > demo_log.txt 2>&1
# The first steps, then only the steps where requests joined or left, then the summary.
!grep '^step' demo_log.txt | head -n 3
!echo '... steps where the batch changed:'
!grep '^step' demo_log.txt | grep -v -- '+0 joined | -0 finished' | sed -n '2,25p'
!sed -n '/output tokens in/,$p' demo_log.txt

## Stage 5: serving benchmarks

The same 200 mixed-length requests through every backend. Offline: everything arrives at t=0 (peak throughput). About 6 minutes.

In [ ]:
!python -m benchmarks.run_bench --backend engine_continuous --rate inf 2>&1 | grep '^num_requests'
!python -m benchmarks.run_bench --backend engine_static --rate inf 2>&1 | grep '^num_requests'
!python -m benchmarks.run_bench --backend hf_static --rate inf 2>&1 | grep '^num_requests'
!python -m benchmarks.run_bench --backend naive --rate inf --num-requests 10 2>&1 | grep '^num_requests'

Online: Poisson arrivals at 2 and 4 requests per second (latency under load). About 11 minutes.

In [ ]:
!for b in engine_continuous engine_static hf_static; do python -m benchmarks.run_bench --backend $b --rate 2 2>&1 | grep '^num_requests'; done
!for b in engine_continuous engine_static; do python -m benchmarks.run_bench --backend $b --rate 4 2>&1 | grep '^num_requests'; done

Real vLLM for comparison. It pins its own PyTorch, so it goes into a separate virtualenv and never touches the engine's environment. About 5 minutes.

In [ ]:
!pip install -q uv
!test -x /content/vllm-venv/bin/python || (uv venv -q -p 3.12 /content/vllm-venv && uv pip install -q -p /content/vllm-venv/bin/python vllm)
!PYTHONPATH=. /content/vllm-venv/bin/python benchmarks/vllm_bench.py > vllm_log.txt 2>&1; grep '^num_requests' vllm_log.txt || tail -n 20 vllm_log.txt

## Stage 6: extras

### Streaming API

Server tests (FastAPI + Server-Sent Events), then the real server under load: many clients at once, each streaming its tokens.

In [ ]:
!python -m pytest tests/test_api.py -q -p no:warnings

In [ ]:
# Start the server in the background, wait until it answers, stream one request, then load-test it.
!nohup python -m uvicorn api.server:app --port 8000 > server_log.txt 2>&1 &
!for i in $(seq 1 180); do curl -sf localhost:8000/health > /dev/null && break; sleep 1; done; curl -s localhost:8000/health
!curl -sN localhost:8000/generate -H 'Content-Type: application/json' -d '{"prompt": "Write one sentence about GPUs.", "max_tokens": 24}' | head -n 4
!python -m api.load_test --concurrency 1 8 32 64 --save 2>&1 | grep -E '^concurrency|Error'
!pkill -f 'uvicorn api.server:app'

### Paged KV cache

The same KV memory (32 x 1024 tokens) split into 16-token blocks, so up to 128 sequences can run at once. Parity tests include a pool small enough to force preemption.

In [ ]:
!python -m pytest tests/test_kv_cache.py tests/test_engine_parity.py tests/test_scheduler.py -q -p no:warnings
!python -m benchmarks.run_bench --backend engine_paged --rate inf --max-num-seqs 128 2>&1 | grep '^num_requests'

### Prefix caching

Every request starts with the same 512-token system prompt. With prefix caching its blocks are computed once and shared. About 3 minutes.

In [ ]:
!for r in inf 4; do for b in engine_paged engine_paged_prefix; do python -m benchmarks.run_bench --backend $b --rate $r --max-num-seqs 128 --shared-prefix 512 2>&1 | grep '^num_requests'; done; done

### CUDA graphs

The decode step recorded once per (batch, KV length) bucket and replayed with a single launch. Tests first: graphed outputs must equal eager outputs. About 5 minutes.

In [ ]:
!python -m pytest tests/test_cuda_graphs.py -q -p no:warnings
!python -m benchmarks.static_batch --backend engine_graphs
!python -m benchmarks.run_bench --backend engine_continuous --rate inf --cuda-graphs 2>&1 | grep '^num_requests'
!python -m benchmarks.run_bench --backend engine_paged --rate inf --max-num-seqs 128 --cuda-graphs 2>&1 | grep '^num_requests'
!python -m benchmarks.run_bench --backend engine_continuous --rate 4 --cuda-graphs 2>&1 | grep '^num_requests'

## Charts

In [ ]:
!python -m benchmarks.plot
import glob
from IPython.display import Image, display
for png in sorted(glob.glob('benchmarks/results/chart*.png')):
    display(Image(png))